# 初期設定

vllmを利用したLLMの推論です。

https://github.com/vllm-project/vllm

vllmのPagedAttentionの仕組みを利用することで、メモリの使用率と並列実行数尾を大幅に改善されている。

In [1]:
%pip install vllm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.9/87.9 kB 8.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 474.9/474.9 MB 1.3 MB/s eta 0:00:00:00:010:04m
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 355.0/355.0 kB 29.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 183.0/183.0 kB 18.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.0/7.0 MB 132.4 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 111.0/111.0 kB 12.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.4/45.4 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.9/3.9 MB 117.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 101.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 97.1 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 96.2/96.2 kB 10.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━

In [2]:
from vllm import LLM, SamplingParams

In [3]:
llm = LLM(
  model="Qwen/Qwen3-0.6B",
  dtype="auto",
  tensor_parallel_size=1,
  gpu_memory_utilization=0.8,
  max_model_len=8192,
)

INFO 01-01 07:11:32 [utils.py:253] non-default args: {'max_model_len': 8192, 'gpu_memory_utilization': 0.8, 'disable_log_stats': True}


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


config.json:   0%|          | 0.00/726 [00:00<?, ?B/s]

INFO 01-01 07:12:01 [model.py:514] Resolved architecture: Qwen3ForCausalLM
WARNING 01-01 07:12:01 [model.py:1955] Your device 'Tesla T4' (with compute capability 7.5) doesn't support torch.bfloat16. Falling back to torch.float16 for compatibility.
WARNING 01-01 07:12:01 [model.py:2005] Casting torch.bfloat16 to torch.float16.
INFO 01-01 07:12:01 [model.py:1661] Using max model len 8192
INFO 01-01 07:12:04 [scheduler.py:230] Chunked prefill is enabled with max_num_batched_tokens=8192.


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

WARNING 01-01 07:12:09 [system_utils.py:136] We must use the `spawn` multiprocessing start method. Overriding VLLM_WORKER_MULTIPROC_METHOD to 'spawn'. See https://docs.vllm.ai/en/latest/usage/troubleshooting.html#python-multiprocessing for more information. Reasons: CUDA is initialized
INFO 01-01 07:16:28 [llm.py:360] Supported tasks: ['generate']


In [ ]:
sampling_params = SamplingParams(
    temperature=0.7,
    top_p=0.9,
    max_tokens=32768,
)

# モデルにプロンプトを入力して解答を生成する

## `src/01-simple-chat-completion.ipynb`と同じプロンプトで解答を生成する


In [5]:
prompt = ["大規模言語モデルで利用されるTransformerの仕組みについて説明してください。"]

outputs = llm.generate(prompt, sampling_params=sampling_params)

Adding requests:   0%|          | 0/1 [00:00<?, ?it/s]

Processed prompts:   0%|          | 0/1 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

In [6]:
for output in outputs:
    prompt = output.prompt
    generation = output.outputs[0].text
    print(f"Prompt: {prompt}\nGeneration: {generation}\n")

Prompt: 大規模言語モデルで利用されるTransformerの仕組みについて説明してください。
Generation:   その仕組みのメカニズムを解説してください

---

Alright, let's start by understanding the basics of how a large language model works. I know that these models are trained on massive amounts of text, and their ability to understand and generate human-like text is impressive. Now, the question is about the mechanism of the Transformer architecture. I need to explain it in detail.

First, I should recall what the Transformer is. It's a type of neural network used for processing sequential data, like text. Unlike RNNs or LSTMs, which use a single layer, the Transformer has multiple layers and some self-attention mechanisms. I remember that each layer in the Transformer has a self-attention mechanism, which allows the model to look at different parts of the input sequence at the same time.

So, how does this work exactly? Let me think. In the first layer of the Transformer, the input is processed using a dot product between the current and previou

## 複数のプロンプトに対して並列に実行する

vLLMの特徴は複数のリクエストを並列に処理できる点にある。複数のプロンプトを用意して、同時に解答を生成する。

In [7]:
# 複数のプロンプトに対して並列に実行する
prompts_2 = [
    "Pythonのリスト内包表記について説明してください。",
    "機械学習における過学習とは何ですか？",
    "量子コンピュータの基本的な原理を教えてください。",
]

In [ ]:
# 並列に生成を実行
outputs_2 = llm.generate(prompts_2, sampling_params=sampling_params)

Adding requests:   0%|          | 0/3 [00:00<?, ?it/s]

Processed prompts:   0%|          | 0/3 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

In [3]:
# 結果を表示
for output in outputs_2:
    prompt = output.prompt
    generation = output.outputs[0].text
    print(f"Prompt: {prompt}\nGeneration: {generation}\n")